# La publicité fait-elle vendre ?

Ce test proposé par Numberly consite en 2 étapes :
1) Comprendre le marché ;
2) Comprendre l’impact de la publicité sur les ventes.

Commençons par faire un point sur les données à disposition.

**Convention de nommage des tables :**
| Niveau | Noms | Contenu | Sert à |
|---|---|---|---|
| A | `retailer`, `programmatic_pub`, `tv_pub`, `socio_demo`, `mapping` | Brut, jamais modifié | Audit uniquement |
| B | `retailer_c`, `prog_c`, `tv_c`, `socio_c`, `mapping_c` | Doublons retirés, types convertis, "unknown" remplacés par NaN | Totaux : CA, budget média, funnel, saisonnalité |
| C | `retailer_id`, `prog_id`, `tv_id`, `socio_id`, `mapping_id` | Niveau B sans les lignes dont l'identifiant est manquant | Tout ce qui passe par un client : jointures, attribution, panier moyen par acheteur |

## 0) Chargement des données

**Bibliothèques nécessaire :**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

**Chargement des données :**

In [ ]:
retailer = pd.read_csv("retailer.csv")
programmatic_pub = pd.read_csv("programmatic_publisher.csv")
tv_pub = pd.read_csv("tv_publisher.csv")
socio_demo = pd.read_csv("socio_demo.csv")
mapping = pd.read_csv("mapping_transac_publisher_tv.csv")

datasets = {
    "retailer": retailer,
    "programmatic_pub": programmatic_pub,
    "tv_pub": tv_pub,
    "socio_demo": socio_demo,
    "mapping": mapping,
}

print("Données chargées :")
for name, df in datasets.items():
    print(f"  {name:<18}: {df.shape}")

## 1) Audit des données

Toutes les vérifications de cette section portent sur les tables brutes (niveau A), sans aucune modification.

### Types des variables :

In [ ]:
for name, df in datasets.items():
    print(f"\n{'=' * 50}\n{name}\n{'=' * 50}")
    df.info()

Point de vigilance : 
- les types de certaines variables ne sont pas toujours corrects (ex : les variables de temps doivent être converties).

### Statistiques de base :

In [ ]:
for name, df in datasets.items():
    print(f"\n{'=' * 50}\n{name}\n{'=' * 50}")
    print(df.describe(include="all").round(2))

Points de vigilance : 
- les identifiants (`customer_id`, `dsp_id`, `device_id`) contiennent tous la valeur "unknown", qui équivaut à une valeur manquante;
- les variables `sales` et `quantity` de la table retailer ont comme valeur minimale 0.

### Valeurs manquantes :

In [ ]:
for name, df in datasets.items():
    missing = pd.DataFrame({
        "Valeurs manquantes": df.isna().sum(),
        "Pourcentage (%)": (df.isna().mean() * 100).round(0).astype(int)
    })
    print(f"\n{'=' * 50}\n{name}\n{'=' * 50}")
    print(missing)

#### Zoom sur la table `retailer`

Chaque ligne est un événement enregistré sur le site du distributeur pour un client et un produit donné.  
Regardons les types d'événements pour comprendre les valeurs manquantes.

In [ ]:
retailer.sample(5)

In [ ]:
events = pd.DataFrame({
    "Lignes": retailer["event_name"].value_counts(),
    "Part (%)": (retailer["event_name"].value_counts(normalize=True) * 100).round(1)
})
print(events)

Trois étapes successives existent :

| `event_name`      | Lignes    | Part   | Signification                        |
|-------------------|-----------|--------|--------------------------------------|
| Product Page View | 6 376 627 | 65   % | Le client consulte une fiche produit |
| Add to cart       | 2 050 111 | 21   % | Il ajoute le produit au panier       |
| Order             | 1 439 311 | 15   % | Il achète                            |

- Variables `sales` et `quantity` :

On ne peut parler de montant et de quantité que s'il y a eu une vente : les valeurs manquantes de `sales` et `quantity` semblent être structurelles. Vérifions-le.

In [ ]:
missing_sales_qt = retailer[retailer[["sales", "quantity"]].isna().any(axis=1)]
print("Événements associés aux valeurs manquantes de sales / quantity :")
print(missing_sales_qt["event_name"].value_counts(dropna=False))

Seuls les événements `Product Page View` et `Add to cart` sont concernés, et aucun `Order` : les manquants sont bien structurels et ne posent pas de problème.  

**Décision :** aucune imputation ; les KPI de vente se calculent sur event_name == "Order".

- Variables `brand` et `product_name` :

In [ ]:
missing_brand = retailer["brand"].isna()
missing_product = retailer["product_name"].isna()
print("Exactement les mêmes lignes :", missing_brand.equals(missing_product))


Ce sont les mêmes lignes qui sont concernées par les valeurs manquantes de ces deux variables. Quand l’un manque, l’autre manque aussi.

In [ ]:
missing_brand_product = retailer[retailer["brand"].isna()]
events = pd.DataFrame({
    "Valeurs manquantes": missing_brand_product["event_name"].value_counts(dropna=False)
})
# Nombre total de lignes par événement dans le dataset complet
events["Total événement"] = retailer["event_name"].value_counts(dropna=False)
# Pourcentage de valeurs manquantes par événement
events["Pourcentage (%)"] = (
    events["Valeurs manquantes"] / events["Total événement"] * 100
).round(1)

print(events)


Les manquants se trouvent dans les trois types d’événements et représentent qu'une faible part de celles-ci.

___________________ 

|>>>

In [ ]:
# Poids dans les ventes
commande = retailer[retailer['event_name'] == 'Order']
poids = commande.groupby(commande['brand'].isna())['sales'].agg(['count', 'sum','mean']).round().astype(int)
poids["part du CA (%)"] = (poids["sum"] / poids["sum"].sum() * 100).round(2)
print(poids)

In [ ]:
# Répartition dans le temps et chez les clients
manq = retailer[retailer["brand"].isna()]
mois_total = retailer["timestamp_utc"].str[:7].value_counts().sort_index()
mois_manq = manq["timestamp_utc"].str[:7].value_counts().sort_index()
print(pd.DataFrame({"manquants": mois_manq, "part (%)": (mois_manq / mois_total * 100).round(2)}))

print("\nClients concernés :", manq["customer_id"].nunique(),
      "sur", retailer["customer_id"].nunique(), "clients distincts")

*Poids dans les ventes. Les 8 776 commandes sans marque pèsent 699 706 € sur un total de 62,43 M€, soit 1,1 % du CA. 
Un point à noter. Ces commandes ont un montant moyen d’environ 80 € par ligne, contre environ 43 € pour les autres. Elles sont donc moins nombreuses mais plus chères que la moyenne. Ça n’invalide rien, mais ça empêche de dire qu’elles sont « comparables aux autres ».
Dans le temps. Les manquants sont présents tous les mois (de janvier à juin 2024), sans pic isolé. Ça exclut un bug ponctuel sur un mois. Ils augmentent de 33 k à 59 k par mois, mais ça peut simplement suivre la hausse de l’activité globale du site.
Chez les clients. 134 172 clients distincts sont concernés pour 277 240 lignes, soit environ 2 lignes par client. Ce n’est donc pas quelques comptes qui posent problème, c’est diffus.*

*à faire: 2. Compare aussi à la population totale : retailer['customer_id'].nunique(), pour dire quelle part des clients est concernée.*

*Marque et produit manquants (277 240 lignes, 2,8 %). Les deux colonnes sont manquantes simultanément, ce qui suggère une absence d’information produit à la source. Ces lignes se répartissent sur tous les types d’événements (3,8 % des vues de page, 1,2 % des ajouts au panier, 0,6 % des commandes), sur l’ensemble de la période (janvier à juin 2024) et sur 134 172 clients différents : il n’y a ni pic temporel ni concentration sur quelques comptes, ce qui écarte un incident ponctuel. Côté ventes, les 8 776 commandes concernées représentent 0,70 M€, soit 1,1 % du CA total. Leur montant moyen par ligne (~80 €) est supérieur à celui des autres commandes (~43 €), ce qui peut indiquer un type de produit particulier non catalogué.*

**Décision :** aucune imputation. Ces lignes sont exclues des analyses par marque et par produit, mais conservées pour les analyses au niveau client et pour le CA total.

|<<
____________________________

#### Zoom sur la table `mapping`

La variable `device_id` a des valeurs manquantes (NaN). Regardons à quoi elles correspondent.

In [ ]:
nan_dev = mapping[mapping["device_id"].isna()]
nan_dev.head()

Les valeurs manquantes semblent concerner les lignes où `customer_id` est inconnu, vérifions si c'est le seul cas :

In [ ]:
print("customer_id des lignes avec device_id manquant :")
print(nan_dev["customer_id"].value_counts(dropna=False))

unknown_cust = mapping[mapping["customer_id"] == "unknown"]
print("\ndevice_id des lignes avec customer_id unknown :")
print(unknown_cust["device_id"].value_counts(dropna=False))

_______________
|>>

Toutes les valeurs manquantes de `device_id` correspondent à un `customer_id` inconnu, et inversement.  
Ces lignes ne relient donc aucun client à une box TV : elles sont inutilisables pour l'attribution.

|<<
_______________


### Identifiants "unknown" :


Dans toutes les tables, l'absence d'identifiant est parfois codée par le texte `"unknown"`. Pandas ne le voit pas comme un manquant : il faut le compter explicitement.

In [ ]:
def count_unknown(datasets):
    id_cols = ["customer_id", "dsp_id", "device_id"]
    results = []
    for name, df in datasets.items():
        for col in id_cols:
            if col in df.columns:
                n_unknown = df[col].astype("string").str.strip().str.lower().eq("unknown").sum()

                results.append({
                    "Table": name,
                    "Variable": col,
                    "Nombre de unknown": n_unknown,
                    "Pourcentage (%)": round(n_unknown / len(df) * 100,2) if len(df) > 0 else 0
                })
    return pd.DataFrame(results)

unknown_summary = count_unknown(datasets)
print(unknown_summary)

Pour les tables `retailer`, `programmatic_pub`, `tv_pub` et `socio_demo`, une ligne "unknown" ne peut pas servir à relier deux tables : à la jointure, tous les "unknown" seraient considérés comme une seule personne, ce qui ferait exploser le nombre de lignes et attribuerait des ventes à des individus qui n'existent pas.

- Pour les totaux : ces lignes sont conservées, l'argent a bien été dépensé.
- Pour toute analyse par client : ces lignes sont exclues.


Une ligne du `mapping` sert à relier un client à ses identifiants média. Elle est donc utile seulement si le customer_id est connu :
- customer_id unknown : ligne inutilisable pour relier une pub à un achat. On l’exclut.
- dsp_id unknown (client connu) : exposition programmatique non mesurable.
- device_id unknown (client connu) : exposition TV non mesurable.

("Non mesurable" ne veut pas dire "non exposé".)

____________
|>>

### Valeurs nulles dans `sales` et `quantity` :

In [ ]:
orders = retailer[retailer["event_name"] == "Order"]
print(f"sales == 0            : {(orders['sales'] == 0).sum()}")
print(f"quantity == 0         : {(orders['quantity'] == 0).sum()}")

nulle = orders[(orders["sales"]==0) | (orders["quantity"]==0)].copy()
# 2. Poids : part des lignes Order concerné
print(f"Part des lignes Order : {len(nulle) /len(orders):.2%}")

print("\n")
cas = pd.crosstab(nulle["sales"] ==0, nulle["quantity"] ==0,rownames=["sales == 0"], colnames=["quantity == 0"])
cas = cas.reindex(
    index=[False, True],
    columns=[False, True],
    fill_value=0
)
print(cas)

# 3. Quels produits / marques ?
print("\n")
print(nulle["product_name"].value_counts().head(10))
print("\n")
print(nulle.merge(orders[["customer_id", "timestamp_utc", "product_name", "brand"]].drop_duplicates(), how="left")["brand"].value_counts(dropna=False))

# 4. Concentrées dans le temps ou chez certains clients ?
print("\n")
nulle["timestamp_utc"] = pd.to_datetime(nulle["timestamp_utc"])
print(nulle["timestamp_utc"].dt.to_period("M").value_counts().sort_index())
print("\n")
print("Clients  distincts  :",  nulle["customer_id"].nunique(), "pour", len(nulle), "lignes")
print("\n")
print(nulle.groupby("customer_id").size().describe())
print("\n")
print(nulle.groupby("customer_id").size().value_counts().sort_index())

Le poids est faible, cela représente moins de 1% des lignes de commande des clients identifiés. Ces anomalies ne semblent ni être concetrées sur une marque, ni sur un produit donc ça semble être des erreurs / des bruits de collecte. Lorsque quantity = 0 et sales = 0, on supprime ces lignes, elles n’apportent aucune information.

In [ ]:
offerts = orders[(orders["sales"] ==0) & (orders["quantity"] >0)].copy()
normales = orders[(orders["sales"] >0) & (orders["quantity"] >0)].copy()

# Prix de référence par produit, calculé sur les lignes normales
normales["pu"] = normales["sales"] / normales["quantity"]
prix_ref = normales.groupby("product_name")["pu"].median().rename("prix_ref")

# Quantités : ressemblent-elles aux lignes normales ?
print(pd.DataFrame({"A": offerts["quantity"].describe(), "réf": normales["quantity"].describe()}).round(2))

# Produits et marques concernés, par rapport à leur poids habituel
poids = pd.DataFrame({"part dans A (%)": offerts["product_name"].value_counts(normalize=True) *100,"part dans réf (%)": normales["product_name"].value_counts(normalize=True) *100,}).sort_values("part dans A (%)", ascending=False).head(10).round(2)
print("\n")
print(poids)
print("\n")
print(offerts["brand"].value_counts(dropna=False, normalize=True).round(3))

# Clients et temps
print("Clients   distincts   :",   offerts["customer_id"].nunique(), "pour", len(offerts), "lignes")
offerts["timestamp_utc"] = pd.to_datetime(offerts["timestamp_utc"])
print(offerts["timestamp_utc"].dt.to_period("M").value_counts().sort_index())

# Ligne seule ou accompagnée d'autres lignes dans la même commande ?
orders["timestamp_utc"] = pd.to_datetime(orders["timestamp_utc"])
taille =orders.groupby(["customer_id", "timestamp_utc"]).size().rename("nb_lignes")
a = offerts.merge(taille.reset_index(), on=["customer_id", "timestamp_utc"])
print("Lignes         par         commande         (cas         A)         :");
print(a["nb_lignes"].value_counts().head())
print("Dont  la  commande  ne  contient  que  cette  ligne  :",  (a["nb_lignes"] ==1).mean().round(3))

Après  étude  je  décide  également  de  supprimer  ces  lignes  pour  les  raisons  suivantes  :  -  elles faussent le comportement d’achat : ça compte comme une vraie commade alors que le montant est nul - elles n’ont pas de snes pour la partie publicité. Un envoi gratuit n’est pas un achat déclenché par une exposition publicitaire. - leur retrait ne coûte rien. Elles représentent moins de 1% des lignes donc le total des ventes ne bouge pas vraiment si on retire ces lignes.6 295 lignes Order (0,4 %) ont un montant nul ; toutes celles dont la quantité est nulle (2 723) en font partie. Les 3 572 restantes ont une quantité de 1 dans la quasi-totalité des cas, sont isolées dans leur commande, et se répartissent uniformément dans le temps (3 252 clients, tous les mois) : aucun motif d’erreur systématique, mais aucun paiement associé. Décision : retrait de toutes ces lignes. Une commande sans montant n’est pas un achat et fausserait le nombre de commandes par client, le taux de réachat et le panier moyen. Leur impact sur le CA est nul.

### Doublons :

In [ ]:
for name, df in datasets.items():
    print(f"{name:<18}: {df.duplicated().sum():>8,} doublons exacts")

On regarde s'il s'agit de vrais doublons ou de lignes volontairement répétées.

**`retailer`**

In [ ]:
dup = retailer[retailer.duplicated(keep=False)]
dup.sort_values(by=retailer.columns.tolist()).head(10)

In [ ]:
print("Lignes impliquées par type d'événement :")
print(dup["event_name"].value_counts())

**`programmatic_pub`**

In [ ]:
programmatic_pub[
    programmatic_pub.duplicated(keep=False)
].sort_values(by=programmatic_pub.columns.tolist()).head(10)

**`tv_pub`**

In [ ]:
tv_pub[
    tv_pub.duplicated(keep=False)
].sort_values(by=tv_pub.columns.tolist()).head(10)

**`mapping`**

In [ ]:
mapping[mapping.duplicated(keep=False)].head()

In [ ]:
mask_vide = (
    mapping["customer_id"].eq("unknown")
    & mapping["dsp_id"].eq("unknown")
    & mapping["device_id"].isna()
)
print("Lignes 'customer unknown + dsp unknown + device NaN' :", mask_vide.sum())
print("Doublons restants hors ces lignes :", mapping[~mask_vide].duplicated().sum())

Pour `retailer`, `programmatic_pub` et `tv_pub`, les doublons sont exacts sur toutes les variables, avec un timestamp identique à la seconde : ils semblent involontaires.

Pour `mapping`, les seuls doublons sont les lignes sans aucune information utile (`customer_id` et `dsp_id` unknown, `device_id` NaN) : on les supprime toutes.

### Synthèse :

| Constat | Décision | Niveau concerné |
|---|---|---|
| `sales` / `quantity` manquants : structurels, seuls les `Order` en ont | Aucune imputation ; filtre `event_name == "Order"` au moment du calcul des KPI de vente | Analyse |
| `brand` / `product_name` manquants | Aucune imputation ; exclus des analyses par marque / produit uniquement | Analyse |
| `device_id` NaN dans `mapping` = `customer_id` unknown | Lignes exclues pour l'attribution | C |
| Identifiants `unknown` | NaN en B ; lignes retirées en C | B puis C |
| Doublons exacts | Supprimés | B |
| Clients connus avec `dsp_id` ou `device_id` manquant | Conservés, marqués « non mesurables » | C |

___________
reprendre ici

## 2) Nettoyage

Ordre des opérations :
1. doublons exacts ;
2. types
3. "unknown" remplacés par NaN.
4. retrait des Order à montant nulle

### Niveau B : tables nettoyées (pour les totaux)

On commence par retirer les doublons.

In [ ]:
def drop_dups(df, name):
    out = df.drop_duplicates(ignore_index=True)
    print(f"{name:<18}: {len(df) - len(out):,} doublons retirés")
    return out

# mapping : on retire d'abord les lignes entièrement vides (customer et dsp unknown, device NaN)
mapping_c = mapping.loc[~mask_vide]
print(f"{'mapping':<18}: {mask_vide.sum():,} lignes entièrement vides retirées")

retailer_c = drop_dups(retailer, "retailer")
prog_c = drop_dups(programmatic_pub, "programmatic_pub")
tv_c = drop_dups(tv_pub, "tv_pub")
mapping_c = drop_dups(mapping_c, "mapping")
socio_c = socio_demo.copy()

Ensuite on convertit les types de variable.

In [ ]:
# Types : dates, coût converti (1 milli-cent = 1/100 000 de l'unité monétaire), catégories
for df in (retailer_c, prog_c, tv_c):
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"])

for df in (prog_c, tv_c):
    df["cost"] = df["cost_milli_cent"] / 100_000

for df, cols in [(retailer_c, ["event_name", "brand"]),
                 (prog_c, ["campaign_name", "device_type"]),
                 (socio_c, ["breed", "age", "income"])]:
    df[cols] = df[cols].astype("category")

Puis on remplace les "unknown" par NaN :

In [ ]:
for df, cols in [(retailer_c, ["customer_id"]),
                 (prog_c, ["dsp_id"]),
                 (tv_c, ["device_id"]),
                 (socio_c, ["customer_id"]),
                 (mapping_c, ["customer_id", "dsp_id", "device_id"])]:
    for c in cols:
        df[c] = df[c].mask(df[c] == "unknown")

Pour finir on retire les lignes des Order à montant nul (quantity == 0 implique sales == 0, une seule règle suffit) :

In [ ]:
masque_zero = (retailer_c["event_name"] == "Order") & (retailer_c["sales"] == 0)
print(f"retailer : {masque_zero.sum():,} lignes Order à montant nul retirées")
retailer_c = retailer_c.loc[~masque_zero]

### Niveau C : tables identifiées (pour les analyses par client)

On retire les lignes dont l'identifiant est manquant. 
Dans `mapping`, on ne retire que les lignes sans `customer_id` : un client connu avec `dsp_id` ou `device_id` manquant est conservé.

In [ ]:
retailer_id = retailer_c.dropna(subset=["customer_id"])
prog_id = prog_c.dropna(subset=["dsp_id"])
tv_id = tv_c.dropna(subset=["device_id"])
socio_id = socio_c.dropna(subset=["customer_id"])

mapping_id = mapping_c.dropna(subset=["customer_id"]).copy()
mapping_id["tv_mesurable"] = mapping_id["device_id"].notna()
mapping_id["dsp_mesurable"] = mapping_id["dsp_id"].notna()

print("Part de clients mesurables :")
print(mapping_id[["tv_mesurable", "dsp_mesurable"]].mean().round(3))

### Tables dérivées, définies une seule fois

In [ ]:
orders_c = retailer_c.loc[retailer_c["event_name"] == "Order"].copy()     # totaux, saisonnalité
orders_id = retailer_id.loc[retailer_id["event_name"] == "Order"].copy()  # analyses par client

### Contrôle des volumes

Nombre de lignes à chaque niveau, pour observer ce qui a été retiré.

In [ ]:
controle = pd.DataFrame({
    "A - brut": [len(retailer), len(programmatic_pub), len(tv_pub), len(socio_demo), len(mapping)],
    "B - nettoyé": [len(retailer_c), len(prog_c), len(tv_c), len(socio_c), len(mapping_c)],
    "C - identifié": [len(retailer_id), len(prog_id), len(tv_id), len(socio_id), len(mapping_id)],
}, index=["retailer", "programmatic_pub", "tv_pub", "socio_demo", "mapping"])

controle["Retiré A -> C (%)"] = ((1 - controle["C - identifié"] / controle["A - brut"]) * 100).round(2)
controle

In [ ]:
# Période couverte par chaque table (doit être cohérente entre ventes et médias)
for name, df in [("retailer", retailer_c), ("programmatic_pub", prog_c), ("tv_pub", tv_c)]:
    print(f"{name:<18}: {df['timestamp_utc'].min()}  ->  {df['timestamp_utc'].max()}")

# Couverture de socio_demo
print("\nClients de retailer présents dans socio_demo :",
      retailer_id["customer_id"].isin(socio_id["customer_id"]).mean().round(4))

## 4) Comprendre le marché

### Marché

Ici nous étudions uniquement les données de la table retailer de niveau B.

In [ ]:
retailer_c = retailer_c.copy()
retailer_c["brand"] = retailer_c["brand"].astype("string").fillna("Non renseignée")
retailer_c["product_name"] = retailer_c["product_name"].astype("string").fillna("Non renseigné")

# --- Période d'étude ---
debut, fin = retailer_c["timestamp_utc"].min(), retailer_c["timestamp_utc"].max()
print(f"Période : {debut} -> {fin} ({(fin - debut).days + 1} jours)")

# --- Nombre de clients (identifiés uniquement) ---
print(f"Clients identifiés        : {retailer_id['customer_id'].nunique():,}")
print(f"Clients ayant commandé    : {orders_c['customer_id'].nunique():,}")
print(f"Lignes sans client connu  : {retailer_c['customer_id'].isna().sum():,}")

# --- Répartition des types d'événements ---
ev = retailer_c["event_name"].value_counts()
print(pd.DataFrame({"Lignes": ev, "Part (%)": (ev / ev.sum() * 100).round(0).astype(int)}))

# --- Répartition des marques (lignes par événement + CA des commandes) ---
marques = pd.crosstab(retailer_c["brand"], retailer_c["event_name"], margins=True, margins_name="Total")
marques["Part des lignes (%)"] = (marques["Total"] / marques.loc["Total", "Total"] * 100).round(0).astype(int)
# Part des commandes de chaque marque parmi toutes les commandes
marques["Part des orders (%)"] = (marques["Order"] / marques.loc["Total", "Order"] * 100).round(0).astype(int)
# Part des commandes parmi tous les événements de chaque marque
marques["Orders / marque (%)"] = (marques["Order"] / marques["Total"] * 100).round(0).astype(int)
print(marques)

ca_marque = orders.groupby("brand", observed=True)["sales"].agg(commandes="count", CA="sum")
ca_marque["Part du CA (%)"] = (ca_marque["CA"] / ca_marque["CA"].sum() * 100).round(0).astype(int)
print(ca_marque)

# --- Nombre de produits par marque ---
print(retailer_c.groupby("brand", observed=True)["product_name"].nunique()
      .rename("Produits distincts"))

# --- CA et quantité totaux (événements Order uniquement) ---
print(f"CA total        : {orders['sales'].sum():,.0f}")
print(f"Quantité totale : {orders['quantity'].sum():,.0f}")

La période étudiée est du 1er janvier 2024 au 30 juin 2024, soit une période de 6 mois.
Le table de données contient 1 354 583 clients identifiés, parmis ceux-là, 502 025 ont passé commande.

Environ 2 tiers des lignes correspondent seulement à une vue de la page du produit. 21% sont des ajouts au paniers et 15% un achat.

Il existe 3 marques : Hills, Prescription Diet et Science Diet. La majorité des lignes concernent Science Diet (80%). Cependant c'est la marque Hills qui a le plus d'achat (parmi les actions sur la marque). Mais de part sa faible présence elle représente seulement 2% du CA total.

Sur 505 produits différents, 18 sont de la marque Hills, 189 de la marque Prescription Diet et 298 de la marque Science Diet.

Sur cette période de 6 mois, le chiffre d'affaire total est de 62 427 795 pour 1 555 998 produits vendus.

#### Temporalité

Observons maintenant de plus près la temporalité.

In [ ]:
# --- Préparation ---
orders_c["mois"] = orders_c["timestamp_utc"].dt.to_period("M")
orders_c["jour"] = orders_c["timestamp_utc"].dt.date
orders_c["jour_sem"] = orders_c["timestamp_utc"].dt.dayofweek
orders_c["heure"] = orders_c["timestamp_utc"].dt.hour

jours_fr = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]

In [ ]:
def agg(col):
    """CA et nombre de commandes par modalité de `col`."""
    return orders_c.groupby(col)["sales"].agg(CA="sum", commandes="count")


def tracer(df, titre, xlabel, kind="bar", rotation=0,
           ylim_ca=None, ylim_commandes=None):
    """Deux panneaux : CA puis nombre de commandes, avec zoom optionnel sur Y."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    for ax, col, ylabel, ylim in zip(
        axes,
        ["CA", "commandes"],
        ["CA", "Nombre de commandes"],
        [ylim_ca, ylim_commandes]
    ):
        df[col].plot(kind=kind, ax=ax, color="#3b6ea5")
        ax.set_title(f"{ylabel} {titre}")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", rotation=rotation)
        ax.grid(axis="y", alpha=0.3)

        if ylim is not None:
            ax.set_ylim(ylim)

    if xlabel == "Heure":
        for ax in axes:
            ax.set_xticks(range(24))
            ax.set_xticklabels([f"{h}h" for h in range(24)], rotation=45)
            
    plt.tight_layout()
    plt.show()

##### Par mois :

In [ ]:
par_mois = agg("mois")
tracer(par_mois, "par mois", "Mois", kind="bar")

In [ ]:
par_mois = agg("mois")
tracer(par_mois, "par mois", "Mois", kind="bar", ylim_ca=(0.95e7, 1.1e7), ylim_commandes=(220000,250000))

Que ce soit en terme de CA ou en nombre de commandes, on observe un creu en avril et des pics en janvier, février et mai.

##### Par jour :

In [ ]:
# --- Par jour (série quotidienne + moyenne mobile sur 7 jours) ---
par_jour = agg("jour")
par_jour.index = pd.to_datetime(par_jour.index)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ["CA", "commandes"]):
    ax.plot(par_jour.index, par_jour[col], alpha=0.4, label="Quotidien")
    ax.plot(par_jour.index, par_jour[col].rolling(7, center=True).mean(), label="Moyenne mobile 7 j")
    ax.set_title(f"{col} par jour")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

##### Par jour de la semaine :

In [ ]:
# --- Par jour de la semaine ---
par_js = agg("jour_sem")
par_js.index = jours_fr
tracer(par_js, "par jour de la semaine", "Jour", kind="bar")

In [ ]:
tracer(par_js, "par jour de la semaine", "Jour", kind="bar", ylim_ca=(8e6,9.5e6),ylim_commandes=(195000,210000))

On observe un creu le week-end (samedi et dimanche).

##### Par heure de la journée

In [ ]:
# --- Par heure de la journée ---
par_heure = agg("heure")
tracer(par_heure, "par heure", "Heure", kind="line")

L'heure est en UTC car le fuseau horaire des clients n'est pas fourni. Le creux étant à 8-10h UTC, c'est cohérent avec la uit aux États-Unis. Cependant cela reste une hypothèse.

Si on fait l'hypothèse que c'est le fuseau horaire de la côte Est des États-Unis, cela donne le graphique suivant.

In [ ]:
# Convertir UTC en heure locale de New York
orders_c["timestamp_ny"] = (
    pd.to_datetime(orders_c["timestamp_utc"], utc=True)
      .dt.tz_convert("America/New_York")
)
orders_c["heure_ny"] = orders_c["timestamp_ny"].dt.hour
# --- Par heure de la journée ---
par_heure = agg("heure_ny")
tracer(par_heure, "par heure", "Heure", kind="line")


Dans le cas de cette hypothèse, on observe un gros creu dans la nuit, deux pics vers 10h et 19h et à nouveau un petit creu en journée, aux alentours de 15h.

______________
revoir ça

In [ ]:
# --- Bonus : panier moyen par ligne de commande selon le mois et le jour de la semaine ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
(par_mois["CA"] / par_mois["commandes"]).plot(kind="bar", ax=axes[0], color="#3b6ea5")
axes[0].set_title("CA moyen par ligne de commande, par mois")
(par_js["CA"] / par_js["commandes"]).plot(kind="bar", ax=axes[1], color="#3b6ea5")
axes[1].set_title("CA moyen par ligne de commande, par jour de la semaine")
for ax in axes:
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

_________________

#### Comportement d'achat

In [ ]:
# --- Une ligne par commande (même client, même timestamp) ---
commandes = (orders_id.groupby(["customer_id", "timestamp_utc"])
                .agg(CA=("sales", "sum"),
                     quantite=("quantity", "sum"),
                     nb_lignes=("sales", "size"))
                .reset_index())
print(f"{len(orders_id):,} lignes Order -> {len(commandes):,} commandes")
print(commandes["nb_lignes"].value_counts().head())

Un client peut apparaitre plusieurs fois dans la base de données retailer_id. S'il a plusieurs lignes à la même heure, je prends comme hypothèse qu'il a avait des produits différents dans son panier mais que cela correspond à une unique commande. Je crée donc une nouvelle base de données avec une ligne par commande afin d'étudier le comportement d'achat.

In [ ]:
# Identifiants des commandes contenant exactement 2 lignes
commandes_2 = commandes.loc[commandes["nb_lignes"] == 2].copy()

print(f"Nombre de commandes à 2 lignes : {len(commandes_2):,}")

# Récupérer les lignes d'origine appartenant aux commandes à 2 lignes
lignes_2 = (
    orders_id.merge(
        commandes_2[["customer_id", "timestamp_utc"]],
        on=["customer_id", "timestamp_utc"],
        how="inner"
    )
    .sort_values(["customer_id", "timestamp_utc"])
)

# Afficher quelques exemples
cols = [
    "customer_id", "timestamp_utc", "brand",
    "product_name", "sales", "quantity"
]

display(lignes_2[cols].head(20))

In [ ]:
# --- Table client ---
clients = (orders_id.groupby("customer_id")
              .agg(CA=("sales", "sum"),
                   quantite=("quantity", "sum"),
                   nb_produits=("product_name", "nunique"),
                   nb_lignes=("sales", "size"))
              .join(commandes.groupby("customer_id").size().rename("nb_commandes"))
              .join(commandes.groupby("customer_id")["timestamp_utc"]
                    .agg(premiere="min", derniere="max")))

clients["panier_moyen"] = clients["CA"] / clients["nb_commandes"]
print(f"Acheteurs identifiés : {len(clients):,}")
print(clients[["CA", "quantite", "nb_produits", "nb_commandes", "panier_moyen"]]
      .describe(percentiles=[.25, .5, .75, .9, .99]).round(2))

In [ ]:
# --- Taux de réachat : part des acheteurs avec au moins 2 commandes ---
taux_reachat = (clients["nb_commandes"] >= 2).mean()
print(f"Taux de réachat (>= 2 commandes) : {taux_reachat:.1%}")

# Variante : au moins 2 jours d'achat distincts (plus robuste si plusieurs commandes à la suite)
jours_achat = commandes.assign(jour=commandes["timestamp_utc"].dt.date).groupby("customer_id")["jour"].nunique()
print(f"Taux de réachat (>= 2 jours distincts) : {(jours_achat >= 2).mean():.1%}")

repartition = clients["nb_commandes"].clip(upper=5).value_counts().sort_index()
repartition.index = [str(i) if i < 5 else "5+" for i in repartition.index]
print(pd.DataFrame({"Clients": repartition, "Part (%)": (repartition / repartition.sum() * 100).round(1)}))

#### Etude du parcours sur le site
On distingue les 3 étapes du parcours d'un client sur le site : voir un produit, l’ajouter au panier, commander. Ces trois étapes sont dans event_name : Product Page View, Add to cart, Order.
Observons le taux de conversion, qui est la part de ceux qui passent à l’étape suivante.

In [ ]:
etapes = ["Product Page View", "Add to cart", "Order"]

funnel = pd.DataFrame({
    "Lignes": retailer_c["event_name"].value_counts().reindex(etapes),
    "Clients uniques": retailer_id.groupby("event_name", observed=True)["customer_id"].nunique().reindex(etapes),
})
for col in ["Lignes", "Clients uniques"]:
    funnel[f"{col} : % de l'étape précédente"] = (funnel[col] / funnel[col].shift(1) * 100).round(1)
    funnel[f"{col} : % de la 1re étape"] = (funnel[col] / funnel[col].iloc[0] * 100).round(1)
funnel

Il y a une subtilité : tous les acheteurs n’ont pas forcément une vue enregistrée dans la période. Vérifie-le, car ça change la lecture des pourcentages :

In [ ]:
vus = set(retailer_id.loc[retailer_id["event_name"] == "Product Page View", "customer_id"])
paniers = set(retailer_id.loc[retailer_id["event_name"] == "Add to cart", "customer_id"])
acheteurs = set(retailer_id.loc[retailer_id["event_name"] == "Order", "customer_id"])

print(f"Vu -> panier (parmi les clients ayant vu)   : {len(vus & paniers) / len(vus):.1%}")
print(f"Panier -> commande (parmi ceux au panier)  : {len(paniers & acheteurs) / len(paniers):.1%}")
print(f"Vu -> commande : {len(vus & acheteurs) / len(vus):.1%}")
print(f"Acheteurs sans aucune vue enregistrée       : {len(acheteurs - vus):,} ({len(acheteurs - vus) / len(acheteurs):.1%})")

- Moins d’un client sur deux qui a consulté un produit en a ajouté un au panier sur la période. C’est l’endroit où l’on perd le plus de monde.
- Huit clients sur dix qui ont mis un produit au panier ont fini par commander. Une fois qu’un client a ajouté un produit au panier, il achète presque toujours.

6,0 % des acheteurs (30 127) n’ont aucune vue enregistrée. C’est peu, mais ils sont hors de l’entonnoir tel que tu le comptes.

Attention cependant. 
- Ce sont des taux par client sur six mois, pas par visite. « 80 % » veut dire « 80 % des clients qui ont un jour ajouté un produit au panier ont un jour commandé », et non « 80 % des paniers aboutissent ». Rien dans les données ne prouve que le panier précède la commande dans le temps. N’écris donc pas « 80 % des paniers sont convertis ».
- Les 6 % d’acheteurs sans vue. Je vois deux explications possibles : la vue a eu lieu avant le 1er janvier, car les données commencent à cette date, ou le client est arrivé sans passer par une fiche produit (recommande directe, lien, application). Ce sont des hypothèses. Tu peux tester la première :

In [ ]:
sans_vue = acheteurs - vus
premiere_cmd = retailer_id[retailer_id["event_name"] == "Order"].groupby("customer_id")["timestamp_utc"].min()
t = premiere_cmd.dt.to_period("M").to_frame("mois")
t["sans_vue"] = t.index.isin(sans_vue)
print((t.groupby("mois")["sans_vue"].mean() * 100).round(1))

La part n'est pas plus haute en janvier qu'en juin (c'est l'inverse), elle est plutôt stable, l'explication est plutôt un parcours qui ne passe pas par la fiche produit


En clients uniques, 44,2 % des clients ayant consulté un produit en ont ajouté un au panier, et 80,1 % de ceux qui ont rempli un panier ont commandé. La principale perte se situe donc entre la consultation et l’ajout au panier. Ces taux sont calculés par client sur six mois, et non par session. 6,0 % des acheteurs n’ont aucune consultation enregistrée : [explication selon ton test]. Ils restent dans le décompte des acheteurs, mais pas dans les taux ci-dessus.

conversion par marque. Le principe est de compter, pour chaque marque, les clients uniques à chaque étape.

In [ ]:
par_marque = (retailer_id.groupby(["brand", "event_name"], observed=True)["customer_id"]
                         .nunique().unstack()[etapes])
par_marque["Vue -> commande (%)"] = (par_marque["Order"] / par_marque["Product Page View"] * 100).round(1)
par_marque["Panier -> commande (%)"] = (par_marque["Order"] / par_marque["Add to cart"] * 100).round(1)
print(par_marque)

Faut-il faire aussi par produit ?

#### Etude de sales et quantity


On commence par créer une nouvelle variable : le prix unitaire.

In [ ]:
# --- Variable prix unitaire (une seule fois) ---
assert (orders_c["quantity"] > 0).all() and (orders_c["sales"] > 0).all()
orders_c["prix_unitaire"] = orders_c["sales"] / orders_c["quantity"]

In [ ]:
print(orders_c[["sales", "quantity", "prix_unitaire"]]
      .describe(percentiles=[.01, .25, .5, .75, .95, .99]).round(2))

print("\nQuantité par ligne (% des lignes) :")
print((orders_c["quantity"].value_counts(normalize=True).sort_index().head(8) * 100).round(2))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

orders_c["sales"].clip(upper=orders_c["sales"].quantile(0.99)).plot(kind="hist", bins=60, ax=axes[0, 0])
axes[0, 0].set_title("sales par ligne (tronqué au 99e centile)")

np.log10(orders_c["sales"]).plot(kind="hist", bins=60, ax=axes[0, 1])
axes[0, 1].set_title("log10(sales)")

orders_c["prix_unitaire"].clip(upper=orders_c["prix_unitaire"].quantile(0.99)).plot(kind="hist", bins=60, ax=axes[1, 1])
axes[1, 1].set_title("Prix unitaire (tronqué au 99e centile)")

plt.tight_layout()
plt.show()

Poursuivons avec le prix unitaire par produit.

In [ ]:
prix_produit = (orders_c.groupby("product_name")
    .agg(lignes=("sales", "size"), quantite=("quantity", "sum"), CA=("sales", "sum"),
         prix_median=("prix_unitaire", "median"), prix_min=("prix_unitaire", "min"),
         prix_max=("prix_unitaire", "max"), prix_std=("prix_unitaire", "std"))
    .assign(ecart_relatif=lambda d: d["prix_std"] / d["prix_median"]))

print(f"{len(prix_produit)} produits")
print(prix_produit.sort_values("CA", ascending=False).head(10).round(2))

# Le prix est-il stable pour un même produit ?
print("\nÉcart-type / médiane, par produit :")
print(prix_produit["ecart_relatif"].describe(percentiles=[.5, .9, .99]).round(3))

# Un produit appartient-il à une seule marque ?
print("\nProduits rattachés à plusieurs marques :",
      (orders_c.groupby("product_name")["brand"].nunique() > 1).sum())

In [ ]:
orders_c["prix_ref"] = orders_c.groupby("product_name")["prix_unitaire"].transform("median")
orders_c["ecart_prix"] = orders_c["prix_unitaire"] / orders_c["prix_ref"] - 1

e = orders_c["ecart_prix"].dropna()
print(f"Lignes à ±1 % du prix de référence : {(e.abs() <= 0.01).mean():.1%}")
print(f"Prix inférieur de plus de 10 %       : {(e < -0.10).mean():.1%}")
print(f"Prix supérieur de plus de 10 %       : {(e > 0.10).mean():.1%}")

Si presque toutes les lignes sont à ±1 %, le prix est un prix catalogue, et les variations de CA viennent des volumes. Si une part notable est nettement en dessous, c’est probablement des promotions, ce qui peut brouiller la partie 2 : une hausse des ventes peut venir d’une promo et non de la publicité. C’est une hypothèse, qu’il faudra confirmer par les dates.

évolution dans le temps et lignes atypiques.

In [ ]:
# Prix médian mensuel des 5 produits qui font le plus de CA
top = prix_produit.nlargest(5, "CA").index
evol = (orders_c[orders_c["product_name"].isin(top)]
        .assign(mois=lambda d: d["timestamp_utc"].dt.to_period("M"))
        .groupby(["mois", "product_name"])["prix_unitaire"].median().unstack())
evol.plot(figsize=(10, 4), marker="o")
plt.title("Prix unitaire médian par mois (top 5 produits en CA)")
plt.ylabel("Prix unitaire"); plt.grid(alpha=0.3); plt.show()

# Gros achats
gros = orders_c[orders_c["quantity"] >= 10]
print(f"Lignes avec quantité >= 10 : {len(gros):,} ({len(gros) / len(orders_c):.2%} des lignes, "
      f"{gros['sales'].sum() / orders_c['sales'].sum():.2%} du CA)")
print(orders_c.nlargest(10, "sales")[["customer_id", "product_name", "sales", "quantity", "prix_unitaire"]])

# Prix par marque
print(orders_c.groupby("brand", observed=True)["prix_unitaire"].describe().round(2))